# 02 — YOLOv5 Fine-Tuning Walkthrough

This notebook accompanies `02-yolov5-architecture-and-finetuning.md`.

It is deliberately **markdown-heavy**: fine-tuning a real YOLOv5 model requires the actual
[Ultralytics YOLOv5](https://github.com/ultralytics/yolov5) repository, a GPU (or a lot of patience
on CPU), and a real labeled image dataset — none of which belong in an offline, CPU-only,
quick-running teaching notebook. Instead, this notebook walks through the **actual commands and
config files** you would use for the real workflow, as annotated examples, so you can read them
side by side with the concepts chapter.

The one thing we *do* execute is the last cell: parsing and validating a sample `data.yaml` — the
config file YOLOv5's `train.py` reads to find your dataset — using Python's `yaml` module. That part
needs no GPU, no dataset, and no network access.

## Step 1 — Environment setup (annotated command, not executed)

```bash
git clone https://github.com/ultralytics/yolov5
cd yolov5
pip install -r requirements.txt
```

This clones the Ultralytics YOLOv5 repo and installs its dependencies (PyTorch, OpenCV, etc.).
On a real Sagemaker training job, this setup step is usually baked into a custom training container
image or a Sagemaker-managed PyTorch framework container, rather than run interactively.

## Step 2 — Converting the AWS Ground Truth manifest to YOLO label format (annotated, not executed)

Chapter 01 showed the AWS Ground Truth **augmented manifest** output format — JSON lines with boxes
in `(top, left, width, height)` pixel format. YOLOv5 needs one `.txt` label file per image, with one
line per object in **normalized center format**:

```
<class_id> <x_center> <y_center> <width> <height>
```

all normalized to `[0, 1]` by dividing by image width/height. A conversion script (run once, offline,
before training) would look conceptually like this:

```python
# convert_groundtruth_to_yolo.py  (illustrative, not executed here)
import json

def convert_manifest_line(line, image_width, image_height):
    record = json.loads(line)
    yolo_lines = []
    for ann in record["bounding-box"]["annotations"]:
        x_center = (ann["left"] + ann["width"] / 2) / image_width
        y_center = (ann["top"] + ann["height"] / 2) / image_height
        w_norm = ann["width"] / image_width
        h_norm = ann["height"] / image_height
        yolo_lines.append(f"{ann['class_id']} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
    return yolo_lines
```

Each image ends up with a matching `<image_name>.txt` file in a `labels/` directory sitting alongside
an `images/` directory with the same filename structure — this pairing convention is what
`data.yaml` (below) points YOLOv5 at.

## Step 3 — `data.yaml`: telling YOLOv5 where the dataset lives

This is the config file `train.py --data data.yaml` reads. For this project it would look like:

```yaml
train: ../datasets/charts/images/train
val: ../datasets/charts/images/val
test: ../datasets/charts/images/test

nc: 1
names: ['chart']
```

- `train` / `val` / `test` — paths to the image directories for each split. YOLOv5 automatically
  looks for a sibling `labels/` directory with matching filenames.
- `nc` — number of classes. This project plausibly used a single `chart` class, though a finer
  taxonomy (`bar_chart`, `line_chart`, `table`) is a reasonable variant.
- `names` — the class names, in class-id order (class 0 = `'chart'`, etc.).

The executable cell below parses and validates exactly this content — checking the required keys are
present and that `nc` matches the length of `names`, a common real-world source of silent
misconfiguration.

In [1]:
import yaml

sample_data_yaml = """
train: ../datasets/charts/images/train
val: ../datasets/charts/images/val
test: ../datasets/charts/images/test

nc: 1
names: ['chart']
"""

config = yaml.safe_load(sample_data_yaml)
print("Parsed data.yaml:")
print(config)

required_keys = {"train", "val", "nc", "names"}
missing = required_keys - set(config.keys())
assert not missing, f"data.yaml is missing required keys: {missing}"
assert config["nc"] == len(config["names"]), "nc must match len(names)"

print(f"\ndata.yaml is valid: {config['nc']} class(es) -> {config['names']}")

Parsed data.yaml:
{'train': '../datasets/charts/images/train', 'val': '../datasets/charts/images/val', 'test': '../datasets/charts/images/test', 'nc': 1, 'names': ['chart']}

data.yaml is valid: 1 class(es) -> ['chart']


## Step 4 — Fine-tuning command (annotated, not executed)

With the dataset and `data.yaml` in place, fine-tuning from COCO-pretrained weights is a single
`train.py` invocation:

```bash
python train.py \
  --data data.yaml \
  --weights yolov5s.pt \
  --img 640 \
  --batch-size 16 \
  --epochs 100 \
  --name chart_graph_detector
```

- `--weights yolov5s.pt` — starts from the small COCO-pretrained YOLOv5 checkpoint rather than random
  initialization. This is the transfer-learning step described in Chapter 02: the backbone's
  general-purpose visual features transfer, and fine-tuning adapts the head (and to a lesser extent
  later backbone layers) to chart/graph-specific patterns.
- `--img 640` — input resolution. As Chapter 02 discusses, bumping this to 960 or 1280 is a real
  lever if small chart details (fine gridlines, small legend text) are being missed, at the cost of
  slower training/inference and a heavier Lambda deployment footprint.
- `--batch-size 16` — constrained by the Sagemaker training instance's GPU memory.
- `--epochs 100` — a reasonable starting point for a fine-tuning run on a moderately sized custom
  dataset; in practice tuned against the validation-set mAP curve to catch overfitting.

On AWS Sagemaker specifically, this training loop runs inside a managed training job (a Sagemaker
`Estimator`, typically using a PyTorch framework container or a custom container with the YOLOv5 repo
baked in), with the dataset staged from S3 and the resulting model artifact written back to S3 at the
end of the job — the artifact that Chapter 04 packages into the Lambda deployment.

## Step 5 — Reading training output (annotated, not executed)

YOLOv5's `train.py` writes a `results.csv` and TensorBoard logs to `runs/train/chart_graph_detector/`,
including per-epoch `precision`, `recall`, `mAP_0.5`, and `mAP_0.5:0.95` on the validation set — the
metrics Chapter 03 unpacks in depth. Watching `recall` (YOLOv5's terminology for what Chapter 03 calls
TPR) and `mAP_0.5:0.95` climb and plateau across epochs is the practical signal for when to stop
training or adjust hyperparameters, well before running the final strict-IOU-threshold evaluation
(IOU=0.85) that produced the resume's headline 96% TPR figure — see
`notebooks/03_iou_map_metrics_from_scratch.ipynb` for a hands-on reproduction of that final
evaluation step.

## Step 6 — Inference command (annotated, not executed)

```bash
python detect.py \
  --weights runs/train/chart_graph_detector/weights/best.pt \
  --source path/to/new_document_images/ \
  --img 640 \
  --conf-thres 0.25 \
  --iou-thres 0.45
```

Note the two thresholds here are both **inference-time** settings, distinct from the **evaluation**
IOU threshold (0.85) discussed in Chapter 03:
- `--conf-thres` filters out low-confidence detections before they're reported at all.
- `--iou-thres` controls Non-Maximum Suppression — how aggressively overlapping candidate boxes
  around the same object get merged into a single reported detection.

This is the `best.pt` weights file that ultimately gets packaged into the Lambda container image in
`notebooks/04_lambda_inference_handler_demo.ipynb` and Chapter 04.